# Tarefa 4 — Ajuste da árvore a partir do erro (ICMPMASTER)

Mesma árvore de decisão (CART), mesmo rótulo e baseline da Tarefa 2, mesmo corte (`particao`). Nada é recalculado com o Período B e o **teste fica fechado**.

**O que muda:** a Tarefa 3 treinou um detector (alvo = classe da própria rodada), e o F1 de 0,9999 veio de a árvore redescobrir a regra do rótulo. O RFC define o preditor como o estado **12 minutos à frente**. Aqui o alvo é a classe da rodada 3 posições depois (3 × 240 s) **no mesmo fluxo e no mesmo bloco**. O rótulo em si continua o da Tarefa 2.

**Ajustes permitidos:** profundidade, mínimo de amostras na folha, critério, poda e colunas novas **relativas ao baseline** (tendência do `z_robusto`, só com rodadas passadas). País, IP, rota e RTT absoluto continuam fora.

## 1. Configuração

In [2]:
import json
from pathlib import Path
import itertools, warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, recall_score
from sklearn.tree import DecisionTreeClassifier
warnings.filterwarnings("ignore")

ENTRADA = Path("../../../data/processed")       # saídas da Tarefa 2, sem alteração
SAIDA3 = Path("saida_tarefa3")                  # árvore da Tarefa 3
SAIDA = Path("saida_tarefa4"); SAIDA.mkdir(exist_ok=True)

CLASSES = ["OK", "RISCO", "FALHA"]
F3 = ["z_robusto", "aumento_pct", "jitter_relativo", "perda_pct",
      "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"]          # as da Tarefa 3
TEND = ["z_delta1", "z_media3_ant", "z_inclin5"]                          # novas (relativas ao baseline)
SENTINELA = -999          # vazio vira -999 (fora do domínio real; nunca 0)
SEED = 42
H = 3                     # 3 rodadas de 240 s = 12 min à frente
MEDIR_TESTE = False

GRID_CRITERIO = ["gini", "entropy"]
GRID_DEPTH = [3, 5, 8, 10, 12]
GRID_LEAF = [1, 20, 50, 100, 200]
GRID_CCP = [0.0, 0.0005, 0.001]      # poda por custo-complexidade

proibidas = {"classe", "fluxo_id", "rtt_ms", "rtt_medio_ms", "mediana", "rota_id", "dst_addr", "pais"}
assert not set(F3 + TEND) & proibidas, "feature proibida!"

## 2. Dados, colunas de tendência e alvo a 12 min

As colunas de tendência usam só rodadas **anteriores ou a atual** do mesmo fluxo (sem olhar o futuro): `z_delta1` = `z_robusto` atual − o da rodada anterior; `z_media3_ant` = média do `z_robusto` das 3 rodadas anteriores (mínimo 2 válidas); `z_inclin5` = inclinação da reta do `z_robusto` nas últimas 5 rodadas (exige as 5 válidas). Vazio vira sentinela.

In [3]:
d = pd.read_csv(ENTRADA / "dataset_rotulado_B.csv", parse_dates=["timestamp"])
d = d.sort_values(["fluxo_id", "timestamp"]).reset_index(drop=True)
g = d.groupby("fluxo_id")["z_robusto"]

z = {k: g.shift(k) for k in range(1, 5)}
d["z_delta1"] = d["z_robusto"] - z[1]
lag3 = pd.concat([z[1], z[2], z[3]], axis=1)
d["z_media3_ant"] = lag3.mean(axis=1).where(lag3.notna().sum(axis=1) >= 2)
cinco = (-2 * z[4] - z[3] + z[1] + 2 * d["z_robusto"]) / 10
d["z_inclin5"] = cinco.where(z[2].notna() & z[1].notna() & z[3].notna() & z[4].notna())
d["z_anterior"] = z[1]                 # só para descrever erros (não é feature)

gc = d.groupby("fluxo_id")
d["y_fut"] = gc["classe"].shift(-H)
dt = (gc["timestamp"].shift(-H) - d["timestamp"]).dt.total_seconds()
mesmo_bloco = gc["particao"].shift(-H) == d["particao"]
d["tem_alvo"] = mesmo_bloco & dt.between(660, 780)     # ~12 min, dentro do bloco, sem buraco

print("Linhas:", len(d), "| fluxos:", d["fluxo_id"].nunique())
print("Sem alvo a 12 min (fim do bloco ou buraco):", int((~d["tem_alvo"]).sum()))
a = d[d["tem_alvo"]]
tr, va = a[a["particao"] == "treino"], a[a["particao"] == "validacao"]
resumo = pd.DataFrame({n: b["y_fut"].value_counts().reindex(CLASSES) for n, b in (("treino", tr), ("validacao", va))}).T
resumo["N"] = resumo.sum(axis=1)
display(resumo)

def X(df, cols): return df[cols].fillna(SENTINELA).to_numpy()
def f1m(y, p): return f1_score(y, p, labels=CLASSES, average="macro", zero_division=0)
def rec(y, p): return dict(zip(CLASSES, recall_score(y, p, labels=CLASSES, average=None, zero_division=0)))

Linhas: 160885 | fluxos: 65
Sem alvo a 12 min (fim do bloco ou buraco): 877


y_fut,OK,RISCO,FALHA,N
treino,62585,4847,12777,80209
validacao,24524,2487,4898,31909


## 3. O que a árvore da Tarefa 3 erra neste alvo (validação)

Aplico a árvore da Tarefa 3, **sem retreinar**, ao alvo a 12 min. A separação curto/longo usa a mediana do baseline, só como descrição (como na Tarefa 3).

In [4]:
t3 = joblib.load(SAIDA3 / "arvore_oficial.joblib")
ficha = pd.read_csv(ENTRADA / "baseline_por_fluxo.csv")[["fluxo_id", "mediana"]]
corte = ficha["mediana"].median()
va = va.merge(ficha, on="fluxo_id", how="left")
va["caminho"] = np.where(va["mediana"] >= corte, "longo", "curto")
va["p3"] = t3.predict(X(va, F3))

cm3 = pd.DataFrame(confusion_matrix(va["y_fut"], va["p3"], labels=CLASSES),
                   index=[f"real_{c}" for c in CLASSES], columns=[f"prev_{c}" for c in CLASSES])
display(cm3)
r3 = rec(va["y_fut"], va["p3"])
print(f"Árvore T3 no alvo 12 min: F1 macro {f1m(va['y_fut'], va['p3']):.4f} | recall", {k: round(v, 4) for k, v in r3.items()})

print("\nPor caminho (curto/longo): acerto e recall de FALHA")
for c, b in va.groupby("caminho"):
    acerto = (b["y_fut"] == b["p3"]).mean()
    rf = ((b["p3"] == "FALHA") & (b["y_fut"] == "FALHA")).sum() / max((b["y_fut"] == "FALHA").sum(), 1)
    print(f"  {c:>5}: acerto {acerto:.3f} | recall FALHA {rf:.3f} | N {len(b)}")

pico = (va["z_robusto"] >= 3.5) & (va["z_anterior"] < 2)          # pico isolado: atual alto, anterior normal
fa = (va["p3"] == "FALHA") & (va["y_fut"] != "FALHA")             # alarme de FALHA que não se confirmou
print(f"\nPicos isolados na validação: {int(pico.sum())} | só {(va.loc[pico, 'y_fut'] == 'FALHA').mean():.1%} viram FALHA 12 min depois")
print(f"Falsos alarmes de FALHA da T3: {int(fa.sum())} | desses, {int((fa & pico).sum())} são pico isolado")
print(f"RISCO que sumiu: real RISCO previsto OK = {int(((va['y_fut']=='RISCO') & (va['p3']=='OK')).sum())} de {int((va['y_fut']=='RISCO').sum())}")

,prev_OK,prev_RISCO,prev_FALHA
real_OK,22356,1041,1127
real_RISCO,986,1131,370
real_FALHA,1193,304,3401


Árvore T3 no alvo 12 min: F1 macro 0.6872 | recall {'OK': np.float64(0.9116), 'RISCO': np.float64(0.4548), 'FALHA': np.float64(0.6944)}

Por caminho (curto/longo): acerto e recall de FALHA
  curto: acerto 0.841 | recall FALHA 0.629 | N 15901
  longo: acerto 0.845 | recall FALHA 0.742 | N 16008

Picos isolados na validação: 862 | só 17.5% viram FALHA 12 min depois
Falsos alarmes de FALHA da T3: 1497 | desses, 710 são pico isolado
RISCO que sumiu: real RISCO previsto OK = 986 de 2487


## 4. Ajustar a árvore (fit só no treino, escolha só na validação)

Duas tentativas, com a mesma grade (critério × profundidade × folha mínima × poda): **A** só hiperparâmetros, com as 8 colunas da Tarefa 3; **B** hiperparâmetros + as 3 colunas de tendência. Escolha pelo maior F1 macro na validação (arredondado a 4 casas) e, em empate, a árvore com menos folhas.

In [5]:
def buscar(cols):
    linhas, melhor = [], None
    for crit, dp, lf, ccp in itertools.product(GRID_CRITERIO, GRID_DEPTH, GRID_LEAF, GRID_CCP):
        m = DecisionTreeClassifier(criterion=crit, max_depth=dp, min_samples_leaf=lf, ccp_alpha=ccp,
                                   random_state=SEED).fit(X(tr, cols), tr["y_fut"])     # fit SÓ no treino
        p = m.predict(X(va, cols))
        f, r = f1m(va["y_fut"], p), rec(va["y_fut"], p)
        linhas.append(dict(criterio=crit, max_depth=dp, min_samples_leaf=lf, ccp_alpha=ccp, f1_macro_val=f,
                           recall_FALHA=r["FALHA"], recall_RISCO=r["RISCO"], folhas=m.get_n_leaves()))
    g = pd.DataFrame(linhas)
    g["f1_r"] = g["f1_macro_val"].round(4)
    g = g.sort_values(["f1_r", "folhas"], ascending=[False, True]).reset_index(drop=True)
    return g.drop(columns="f1_r")

gA, gB = buscar(F3), buscar(F3 + TEND)
gA.to_csv(SAIDA / "grade_A_so_hiperparametros.csv", index=False)
gB.to_csv(SAIDA / "grade_B_com_tendencia.csv", index=False)
print("A) só hiperparâmetros (8 colunas):"); display(gA.head(3).round(4))
print("B) hiperparâmetros + tendência:"); display(gB.head(3).round(4))

usar_B = gB.loc[0, "f1_macro_val"] > gA.loc[0, "f1_macro_val"] + 1e-4
cols_final = F3 + TEND if usar_B else F3
best = (gB if usar_B else gA).loc[0]
print("\nEscolhida:", "B (com tendência)" if usar_B else "A (só hiperparâmetros)")

A) só hiperparâmetros (8 colunas):


,criterio,max_depth,min_samples_leaf,ccp_alpha,f1_macro_val,recall_FALHA,recall_RISCO,folhas
0,gini,8,1,0.0005,0.6953,0.646,0.3422,20
1,gini,8,20,0.0005,0.6953,0.646,0.3422,20
2,gini,8,50,0.0005,0.6953,0.646,0.3422,20


B) hiperparâmetros + tendência:


,criterio,max_depth,min_samples_leaf,ccp_alpha,f1_macro_val,recall_FALHA,recall_RISCO,folhas
0,gini,8,1,0.0005,0.7143,0.6825,0.3422,20
1,gini,8,20,0.0005,0.7143,0.6825,0.3422,20
2,gini,8,50,0.0005,0.7143,0.6825,0.3422,20



Escolhida: B (com tendência)


## 5. Árvore ajustada, comparação e regras

In [6]:
dp = None if pd.isna(best.max_depth) else int(best.max_depth)
modelo = DecisionTreeClassifier(criterion=best.criterio, max_depth=dp, min_samples_leaf=int(best.min_samples_leaf),
                                ccp_alpha=float(best.ccp_alpha), random_state=SEED).fit(X(tr, cols_final), tr["y_fut"])
joblib.dump(modelo, SAIDA / "arvore_ajustada.joblib")
va["p4"] = modelo.predict(X(va, cols_final))

print(f"Critério {best.criterio} | max_depth {dp} | min_samples_leaf {int(best.min_samples_leaf)} | "
      f"ccp_alpha {best.ccp_alpha} | profundidade obtida {modelo.get_depth()} | folhas {modelo.get_n_leaves()} | semente {SEED}")
print("Colunas:", cols_final)

r4 = rec(va["y_fut"], va["p4"])
tabela = pd.DataFrame({
    "F1 macro": [f1m(va["y_fut"], va["p3"]), f1m(va["y_fut"], va["p4"])],
    "Recall de FALHA": [r3["FALHA"], r4["FALHA"]],
    "Recall de RISCO": [r3["RISCO"], r4["RISCO"]]},
    index=["Árvore da Tarefa 3 (sem retreinar)", "Árvore da Tarefa 4"]).round(4)
tabela.to_csv(SAIDA / "tabela_tarefa3_tarefa4.csv")
display(tabela)

cm4 = pd.DataFrame(confusion_matrix(va["y_fut"], va["p4"], labels=CLASSES),
                   index=[f"real_{c}" for c in CLASSES], columns=[f"prev_{c}" for c in CLASSES])
cm4.to_csv(SAIDA / "matriz_validacao.csv")
print("Matriz 3×3 da árvore da Tarefa 4 (validação; linhas = real, colunas = previsto):"); display(cm4)

fa4 = (va["p4"] == "FALHA") & (va["y_fut"] != "FALHA")
print(f"Falsos alarmes de FALHA: T3 {int(fa.sum())} -> T4 {int(fa4.sum())} | "
      f"RISCO previsto OK: T3 {int(((va['y_fut']=='RISCO') & (va['p3']=='OK')).sum())} -> "
      f"T4 {int(((va['y_fut']=='RISCO') & (va['p4']=='OK')).sum())}")

def regras(m, cols):
    t, out = m.tree_, []
    def desce(no, cond):
        if t.children_left[no] == -1:
            v = t.value[no][0]; k = int(v.argmax())
            out.append((cond, m.classes_[k], int(t.n_node_samples[no]), v[k] / v.sum()))
            return
        nome, lim = cols[t.feature[no]], t.threshold[no]
        desce(t.children_left[no], cond + [f"{nome} <= {lim:.3f}"])
        desce(t.children_right[no], cond + [f"{nome} > {lim:.3f}"])
    desce(0, [])
    return sorted(out, key=lambda r: -r[2])

R = regras(modelo, cols_final)
txt = [f"se {' e '.join(c)} então {k}  (n={n}, pureza={p:.0%})" for c, k, n, p in R]
(SAIDA / "regras_arvore.txt").write_text("\n".join(txt), encoding="utf-8")
print(f"\nFolhas: {len(R)}. Maior folha de cada classe (n no treino; -999 = valor ausente):")
for c in CLASSES:
    r = next((r for r in R if r[1] == c), None)
    if r: print(f"- {c}: se {' e '.join(r[0])} (n={r[2]}, pureza={r[3]:.0%})")

imp = pd.Series(modelo.feature_importances_, index=cols_final).sort_values(ascending=False).round(3)
print("\nImportância:", imp[imp > 0].to_dict())

Critério gini | max_depth 8 | min_samples_leaf 1 | ccp_alpha 0.0005 | profundidade obtida 8 | folhas 20 | semente 42
Colunas: ['z_robusto', 'aumento_pct', 'jitter_relativo', 'perda_pct', 'timeout_atual', 'n5_timeout', 'n5_aumento80', 'n5_risco', 'z_delta1', 'z_media3_ant', 'z_inclin5']


,F1 macro,Recall de FALHA,Recall de RISCO
Árvore da Tarefa 3 (sem retreinar),0.6872,0.6944,0.4548
Árvore da Tarefa 4,0.7143,0.6825,0.3422


Matriz 3×3 da árvore da Tarefa 4 (validação; linhas = real, colunas = previsto):


,prev_OK,prev_RISCO,prev_FALHA
real_OK,23887,195,442
real_RISCO,1345,851,291
real_FALHA,1424,131,3343


Falsos alarmes de FALHA: T3 1497 -> T4 733 | RISCO previsto OK: T3 986 -> T4 1345

Folhas: 20. Maior folha de cada classe (n no treino; -999 = valor ausente):
- OK: se z_robusto <= 4.008 e n5_risco <= 3.500 e n5_aumento80 <= 0.500 e n5_timeout <= 0.500 e n5_risco <= 1.500 e z_delta1 > -2.408 e z_robusto <= 3.062 e n5_risco <= 0.500 (n=38596, pureza=96%)
- RISCO: se z_robusto <= 4.008 e n5_risco > 3.500 e n5_risco > 4.500 (n=1507, pureza=91%)
- FALHA: se z_robusto > 4.008 e z_media3_ant > 2.488 e z_inclin5 <= 0.990 (n=7868, pureza=96%)

Importância: {'z_robusto': 0.611, 'n5_risco': 0.174, 'z_media3_ant': 0.117, 'n5_aumento80': 0.042, 'n5_timeout': 0.025, 'z_inclin5': 0.012, 'z_delta1': 0.01, 'aumento_pct': 0.01}


## 6. Registro final (o que a Tarefa 5 precisa) e teste fechado

In [7]:
cfg = dict(criterio=best.criterio, max_depth=dp, min_samples_leaf=int(best.min_samples_leaf),
           ccp_alpha=float(best.ccp_alpha), seed=SEED, horizonte_rodadas=H, horizonte_min=H * 4,
           features=cols_final, f1_macro_val=float(tabela.iloc[1, 0]),
           recall_falha_val=float(r4["FALHA"]), recall_risco_val=float(r4["RISCO"]))
(SAIDA / "config_modelo.json").write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding="utf-8")
print("Arquivos em", SAIDA.resolve().name, ":", sorted(p.name for p in SAIDA.iterdir()))
print("Teste não medido nesta tarefa (fechado até a Tarefa 5)." if not MEDIR_TESTE else "ATENÇÃO: teste aberto.")

Arquivos em saida_tarefa4 : ['arvore_ajustada.joblib', 'config_modelo.json', 'grade_A_so_hiperparametros.csv', 'grade_B_com_tendencia.csv', 'matriz_validacao.csv', 'regras_arvore.txt', 'tabela_tarefa3_tarefa4.csv']
Teste não medido nesta tarefa (fechado até a Tarefa 5).
